# 🔎 Georgia RAG — pipeline step by step

This notebook lets you walk through the whole pipeline by hand and inspect what comes out at each stage:

1. Check `.env` and settings
2. (optional) Fetch chat history
3. Raw messages
3b. Spam cleaning (money / drugs / ads / pets; questions are kept)
4. Chunking (you can tweak parameters)
4b. Chunking research — sweep parameters & custom context
4c. Knowledge distillation (threads → Q&A) ← prototype B
5. Indexing
6. Retrieval
7. RAG answer
8. Debug: which prompt is actually sent to GPT

> Run: `uv run jupyter lab` from the project root, then open `notebooks/explore.ipynb`.

> Note: query strings stay in Russian on purpose — they must match the Russian content of the chats.

In [153]:
# So that `import config` / `src.*` work from the notebooks/ folder, move to the project root
import os, sys
from pathlib import Path
if Path.cwd().name == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())
print("Working directory:", os.getcwd())

# Autoreload: edits in src/*.py are picked up without restarting the kernel
%load_ext autoreload
%autoreload 2

Working directory: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## 1. Check `.env` and settings

Make sure the keys were picked up (values are masked).

In [154]:
import importlib, config
importlib.reload(config)

def mask(v):
    return (v[:4] + "…" + str(len(v)) + " chars") if v else "❌ not set"

print("OPENAI_API_KEY  :", mask(config.OPENAI_API_KEY))
print("BOT_TOKEN       :", mask(config.BOT_TOKEN))
print("TELEGRAM_API_ID :", config.TELEGRAM_API_ID or "❌ not set")
print("TELEGRAM_PHONE  :", config.TELEGRAM_PHONE or "❌ not set")
print()
print("Chats:", [c["username"] for c in config.CHATS])
print("Models:", config.EMBED_MODEL, "|", config.CHAT_MODEL)
print("Chunking: max gap =", config.CHUNK_MAX_GAP_MINUTES, "min, max size =", config.CHUNK_MAX_CHARS, "chars")

OPENAI_API_KEY  : sk-p…164 chars
BOT_TOKEN       : 8624…46 chars
TELEGRAM_API_ID : 38918746
TELEGRAM_PHONE  : +995XXXXXXXXX

Chats: ['helpgeorgia', 'ipgeorgiachat']
Models: text-embedding-3-small | gpt-4o-mini
Chunking: max gap = 10 min, max size = 1500 chars


## 2. (optional) Fetch chat history

If you already ran `uv run python -m src.ingest` in the terminal — skip this step.

The first run will ask for the confirmation code from Telegram (entered right in the notebook). After authorization a session file is created, so the code won't be needed again.

In [155]:
# Uncomment to fetch history directly from the notebook:
#
# from telethon import TelegramClient
# from src.ingest import ingest_chat
#
# client = TelegramClient("georgia_ingest", int(config.TELEGRAM_API_ID), config.TELEGRAM_API_HASH)
# await client.start(phone=config.TELEGRAM_PHONE or None)
# for chat in config.CHATS:
#     await ingest_chat(client, chat)
# await client.disconnect()

## 3. Raw messages

Look at what was fetched: how many messages and what they look like.

In [156]:
from src.preprocess import _load_raw

username = config.CHATS[0]["username"]
raw_path = config.RAW_DIR / f"{username}.jsonl"
print("File:", raw_path, "| exists:", raw_path.exists())

if raw_path.exists():
    msgs = _load_raw(raw_path)
    print("Total messages:", len(msgs))
    print("\nLast 5:")
    for m in msgs[-5:]:
        sender = m.get("sender") or "Anonymous"
        print(f"  [{m['date'][:16]}] {sender}: {m['text'][:90]}")
else:
    print("Fetch the history first (step 2 or `uv run python -m src.ingest`).")

File: /Users/aleksandra/Documents/GitHub/telegram-georgia-rag/data/raw/helpgeorgia.jsonl | exists: True
Total messages: 2724

Last 5:
  [2026-09-05T10:45] Stanislav: Тбилиси, переезжаем в пустую квартиру, где почти ничего нет. Может, кто-нибудь отдает нену
  [2026-09-06T10:30] Mariam: Avosend тоже работает
  [2026-09-07T09:43] Mariam: Здравствуйте всем. Подскажите пожалуйста, есть ли в Батуми магазины для беременных? Это дл
  [2026-09-10T09:57] Настасья: Здравствуйте! 👋

Подскажите, пожалуйста, кто-нибудь получал справку/подтверждение о том, ч
  [2026-09-10T10:49] Рома: здоавствуйте, подскажите где найти рускоговорящего юриста в грузии?


## 3b. Spam cleaning

Before chunking we drop spam: "quick money" schemes, veiled drug ads from darkstores, self-promo / advertising, and pet-rehoming ads. **Questions and advice are always kept** (a guard skips matches that look like "подскажите / где можно / кто знает ..."). Patterns live in `src/spam.py` (`SPAM_PATTERNS`) — eyeball the removed examples below and tune them. This cell sets `msgs` to the cleaned list for everything that follows.

> In the pipeline this runs automatically (toggle with `config.FILTER_SPAM`).

In [157]:
from src.spam import filter_spam
from collections import Counter

clean, removed = filter_spam(msgs)
print(f"{len(msgs)} messages -> {len(clean)} clean, {len(removed)} spam removed")
print("by category:", dict(Counter(r["spam"] for r in removed)))

# A few examples of what got removed — eyeball these for false positives:
for r in removed:
    print(f"  [{r['spam']}] {r['text'][:100]}")

# Use the cleaned messages for everything below (chunking research, etc.):
msgs = clean

2724 messages -> 2618 clean, 106 spam removed
by category: {'pets': 86, 'ads': 15, 'drugs': 5}
  [pets] ✨ Ищем заботливых хозяев для нашей милой кошечки Киры! ✨

Кира – настоящая пушистая радость, обожает
  [pets] Здравствуйте. Котята ищут дом. Уличная кошка родила в подъезде. Котята обработаны, знают лоток, едят
  [pets] Пишите @ yuliia_animals чтобы забрать малышку к себе 😍

Бесплатно в Добрые Руки ✨❤️
  [pets] Меня зовут Буся! 😌 Мне полгода. В мае меня нашли в подвале с травмированной лапкой: от ампутации я б
  [pets] Самый нежный в мире котенок Люда ищет дом ❤️

Она ласковая, мурчит и ложится на руки даже к незнаком
  [pets] Очаровательные котята, метисы британцев ищут любящие семьи ❤️

Их нашли на мусорке сразу всех вместе
  [pets] 🐾 Щенячий патруль в поисках заботливых хозяев! 🐾

Внимание, внимание! Наш отряд хвостатых непосед го
  [pets] Эриус — ваше лекарство от аллергии на осеннюю хандру 😊

Настала осень, пришла пора искать себе тепло
  [pets] Роскошная кошка Туся, победившая 

## 4. Chunking — how messages are merged into dialogs

Build chunks and inspect the result. The `CHUNK_MAX_GAP_MINUTES` and `CHUNK_MAX_CHARS` parameters can be changed right here to compare.

In [158]:
# from src.preprocess import chunk_messages

# # Feel free to experiment with the parameters:
# # config.CHUNK_MAX_GAP_MINUTES = 15
# # config.CHUNK_MAX_CHARS = 2000

# chunks = chunk_messages(msgs)
# sizes = [len(c["text"]) for c in chunks]
# print(f"{len(msgs)} messages -> {len(chunks)} chunks")
# if sizes:
#     print(f"Chunk size (chars): min {min(sizes)}, avg {sum(sizes)//len(sizes)}, max {max(sizes)}")

In [159]:
# # Inspect 3 random chunks in full
# import random
# for c in random.sample(chunks, min(3, len(chunks))):
#     print("=" * 70)
#     print(c["link"], "| messages", c["first_msg_id"], "-", c["last_msg_id"])
#     print(c["text"])

## 4b. Chunking research — sweep parameters & try custom context

All knobs are **arguments** to `chunk_messages`, so you can compare variants without editing `config` or restarting the kernel:

- `gap_minutes` — time window for grouping messages
- `max_chars` — chunk body size cap
- `context_max_chars` — cap on the quoted reply context
- `whole_burst` — pull the whole conversation around each ancestor (`True`) or just the single message (`False`)
- `chain_max` — how far up the reply chain to walk
- `context_fn` — full override to experiment with brand-new context logic

> Run the **"3. Raw messages"** cell above first so that `msgs` is loaded.

In [160]:
# from src.preprocess import chunk_messages, ancestor_context

# def preview(msgs, n=3, max_show=900, **params):
#     """Build chunks with the given params and show stats + a few chunks that
#     actually carry reply context (the interesting ones)."""
#     chunks = chunk_messages(msgs, **params)
#     sizes = [len(c["text"]) for c in chunks]
#     with_ctx = [c for c in chunks if "↪" in c["text"]]
#     avg = sum(sizes) // len(sizes) if sizes else 0
#     print(f"params: {params}")
#     print(f"  {len(chunks)} chunks | avg {avg} chars | max {max(sizes) if sizes else 0} | with reply-context: {len(with_ctx)}")
#     # for c in with_ctx[:n]:
#     #     print("-" * 70)
#     #     print(c["link"])
#     #     print(c["text"][:max_show])
#     for c in chunks[:3]:
#         print(c["link"]); print(c["text"][:500]); print("-"*60)
#     return chunks

# # One configuration to start from:
# _ = preview(
#     msgs, 
#     n=10, 
#     gap_minutes=10, 
#     whole_burst=True, 
#     max_chars=2000, 
#     context_max_chars=0
#     )

In [161]:
# import itertools

# print(f"{'gap':>4} {'ctx_max':>8} {'whole':>6} | {'chunks':>7} {'avg':>5} {'w/ctx':>6}")
# for gap, ctx_max, whole in itertools.product([5, 10, 20], [800, 2000, 4000], [True, False]):
#     chunks = chunk_messages(msgs, gap_minutes=gap, context_max_chars=ctx_max, whole_burst=whole)
#     sizes = [len(c["text"]) for c in chunks]
#     avg = sum(sizes) // len(sizes) if sizes else 0
#     n_ctx = sum("↪" in c["text"] for c in chunks)
#     print(f"{gap:>4} {ctx_max:>8} {str(whole):>6} | {len(chunks):>7} {avg:>5} {n_ctx:>6}")

### Experiment: custom `context_fn`

`context_fn(buf, by_id, burst_map) -> list[dict]` returns the messages to quote as context. Define your own right here (no autoreload needed — it lives in the notebook) and pass it via `context_fn=`. When you pass `context_fn`, the `whole_burst` / `context_max_chars` / `chain_max` args are ignored.

In [162]:
# def neighbours_around_ancestors(buf, by_id, burst_map, n=2, cap=2000):
#     """For each reply parent outside the chunk, grab ±n messages around it by
#     position (ignoring time). Pure notebook experiment — edit freely."""
#     ids_sorted = sorted(by_id)
#     pos = {mid: i for i, mid in enumerate(ids_sorted)}
#     body = {m["msg_id"] for m in buf}
#     seen, chars = set(), 0
#     for m in buf:
#         p = m.get("reply_to")
#         if not p or p not in pos:
#             continue
#         i = pos[p]
#         for j in range(max(0, i - n), min(len(ids_sorted), i + n + 1)):
#             mid = ids_sorted[j]
#             if mid in body or mid in seen:
#                 continue
#             msg = by_id[mid]
#             if chars + len(msg["text"]) > cap:
#                 continue
#             seen.add(mid); chars += len(msg["text"])
#     return [by_id[i] for i in sorted(seen)]

# print("=== default (whole burst around ancestors) ===")
# _ = preview(msgs, n=1)
# print("\n=== custom: ±2 neighbours around ancestors (by position) ===")
# _ = preview(msgs, n=1, context_fn=lambda b, i, m: neighbours_around_ancestors(b, i, m, n=2))

## 4c. Knowledge distillation (prototype B)

Instead of indexing raw chunks, group messages into **threads** (reply links + time-neighbours) and let the LLM distill each thread into reusable **question → answer** knowledge, linked to the thread's root message. Threads with no useful knowledge are skipped.

⚠️ Distillation calls `gpt-4o-mini` once per thread — it costs tokens. Start with a small `limit` and eyeball the result before scaling up. Patterns and the prompt live in `src/knowledge.py`; thread building in `src/threads.py`.

In [163]:
from src.threads import build_threads

threads = build_threads(msgs)   # msgs = cleaned messages from step 3b
sizes = sorted((len(t) for t in threads), reverse=True)
multi = [s for s in sizes if s > 1]
print(f"{len(msgs)} messages -> {len(threads)} threads ({len(multi)} multi-message)")
print("top thread sizes:", sizes[:10])

# Look at one multi-message thread in full:
for t in threads:
    if len(t) >= 5:
        print("\n--- thread root", t[0]["link"], f"({len(t)} msgs) ---")
        for m in t:
            print(f"  {(m.get('sender') or '?')[:12]:12} | {m['text'][:60]}")
        break

2618 messages -> 1796 threads (349 multi-message)
top thread sizes: [13, 12, 11, 11, 11, 10, 10, 9, 9, 9]

--- thread root https://t.me/helpgeorgia/305564 (6 msgs) ---
  Omni         | Добрый вечер! Подскажите по заказу с Aliexpress - как удобне
  Tati         | С Китая в основном заказы с Таобао, а не с АлиЭкспресс. АлиЭ
  Omni         | То, что мне нужно не нашёл. На Temu тоже проверял. Нужно име
  Tati         | Там можно напрямую, на почту приходит, через неопределенное 
  Omni         | 3 месяца? )  ладно, понял, спасибо всё равно.
  Leila        | Тбилиси!Срочно нужна помощь по пристройству,котенок 2,5 меся


In [164]:
# Preview WHICH threads "first 20" actually means — before spending any tokens.
# select_threads() is the exact same selection distill_chat() uses internally
# (ascending root_msg_id, after dropping threads whose last message is older
# than config.INGEST_SINCE — see src/knowledge.py).
from src.knowledge import select_threads

preview_threads = select_threads(
    config.CHATS[0]["username"],
    limit=20,
    min_thread_size=2,
)
print(f"{len(preview_threads)} threads selected\n")
for t in preview_threads:
    root = t[0]
    print(f"root={root['msg_id']}  {root['date'][:10]}  ({len(t)} msg)  {root['link']}")


[knowledge] helpgeorgia: skipped 148 threads older than 2025-03-01
20 threads selected

root=308018  2025-02-27  (6 msg)  https://t.me/helpgeorgia/308018
root=308040  2025-03-05  (2 msg)  https://t.me/helpgeorgia/308040
root=308045  2025-03-06  (2 msg)  https://t.me/helpgeorgia/308045
root=308066  2025-03-10  (8 msg)  https://t.me/helpgeorgia/308066
root=308102  2025-03-13  (3 msg)  https://t.me/helpgeorgia/308102
root=308108  2025-03-14  (2 msg)  https://t.me/helpgeorgia/308108
root=308111  2025-03-14  (5 msg)  https://t.me/helpgeorgia/308111
root=308115  2025-03-15  (3 msg)  https://t.me/helpgeorgia/308115
root=308164  2025-03-19  (2 msg)  https://t.me/helpgeorgia/308164
root=308171  2025-03-20  (4 msg)  https://t.me/helpgeorgia/308171
root=308206  2025-03-22  (5 msg)  https://t.me/helpgeorgia/308206
root=308210  2025-03-23  (4 msg)  https://t.me/helpgeorgia/308210
root=308267  2025-03-30  (2 msg)  https://t.me/helpgeorgia/308267
root=308284  2025-04-01  (5 msg)  https://t.me/helpgeo

In [165]:
# ⚠️ Spends OpenAI tokens. Distill the first N multi-message threads to eyeball quality.
from src.knowledge import distill_chat

knowledge = distill_chat(
    config.CHATS[0]["username"],
    limit=20,            # only the first 20 threads (cheap prototype)
    min_thread_size=2,   # discussions only; set 1 to also distill standalone facts
    write=False,         # don't write the jsonl yet, just inspect
)
print(f"\nGot {len(knowledge)} knowledge items from up to 20 threads")

[knowledge] helpgeorgia: skipped 148 threads older than 2025-03-01
[knowledge] helpgeorgia: 20/20 threads -> 13 items

Got 13 knowledge items from up to 20 threads


In [166]:
# View the distilled knowledge — eyeball quality / hallucinations / provenance
for k in knowledge[:15]:
    print("Q:", k["question"])
    print("A:", k["answer"])
    print("  ↳", k["root_link"])
    print("-" * 70)

Q: Где в Тбилиси делается справка о несудимости?
A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Как записаться в сферу интересов РФ для оформления справки о несудимости в РФ?
A: Лучше всего позвонить в соответствующее учреждение, так как правила могут меняться.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Можно ли заказать справку о несудимости через госуслуги?
A: Да, справку о несудимости можно заказать через госуслуги, не нужно никуда ехать.
  ↳ https://t.me/helpgeorgia/308018
----------------------------------------------------------------------
Q: Сколько стоит обмотать чемодан плёнкой в аэропорту?
A: Около 30 лари.
  ↳ https://t.me/helpgeorgia/308045
----------------------------------------------------------------------
Q: Где в Тби

In [167]:
# "Было -> стало": для каждого выбранного треда — исходный текст И то, что LLM из него извлёк.
# Треды сопоставляются с юнитами знаний по root_msg_id.
from src.knowledge import _thread_text
from collections import defaultdict

by_root = defaultdict(list)
for k in knowledge:
    by_root[k["root_msg_id"]].append(k)

for t in preview_threads:
    root = t[0]
    units = by_root.get(root["msg_id"], [])
    print("=" * 70)
    print(f"ТРЕД {root['link']}  ({len(t)} msg)")
    print("--- было (исходный тред) ---")
    print(_thread_text(t)[:800])
    print(f"--- стало ({len(units)} извлечённых пар) ---")
    if not units:
        print("  (ничего не извлечено)")
    for u in units:
        print(f"  Q: {u['question']}")
        print(f"  A: {u['answer']}")
        print(f"  type: {u['type']}  city: {u.get('city')}")
    print()


ТРЕД https://t.me/helpgeorgia/308018  (6 msg)
--- было (исходный тред) ---
Alex: Ребят, а где в Тбилиси делается справка о несудимости? 🥸
Аноним: https://t.me/nlevshitstelegram/19384
Alex: А не подскажете, как записаться в сферу интересов РФ для оформления справки о несудимости именно в РФ? 🇷🇺
Аноним: Позвоните им) Правила могут меняться, так будет самое надёжное
Andrew: На госуслугах я заказывал эту справку. Никуда ехать не приходилось
Nadezhda P.: Апостиль был на справке ?
--- стало (3 извлечённых пар) ---
  Q: Где в Тбилиси делается справка о несудимости?
  A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
  type: vote_based
  Q: Как записаться в сферу интересов РФ для оформления справки о несудимости в РФ?
  A: Лучше всего позвонить в соответствующее учреждение, так как правила могут меняться.
  type: vote_based
  Q: Можно ли заказать справку о несудимости через госуслуги?
  A: Да, справку о несудимости можн

## 4d. Validate the distillation (LLM judge)

Run the same LLM-judge harness used for the full base (`src/eval_knowledge.py`) on just these 20 threads — cheap, and lets you see exactly what the judge outputs and whether it makes sense before trusting it on 220+ units.

In [168]:
# ⚠️ Spends OpenAI tokens (config.JUDGE_MODEL, ~1 call per thread — cheap for 20 threads).
from src.eval_knowledge import judge_units

threads_by_root = {t[0]["msg_id"]: t for t in preview_threads}
judged = judge_units(knowledge, threads_by_root)

# Три колонки в одном виде: ТРЕД -> дистиллировано -> вердикт судьи, сгруппировано по треду.
by_root_j = defaultdict(list)
for j in judged:
    by_root_j[j["root_msg_id"]].append(j)

for t in preview_threads:
    root = t[0]
    js = by_root_j.get(root["msg_id"], [])
    if not js:
        continue  # ничего не дистиллировано из этого треда — нечего судить
    print("=" * 70)
    print(f"ТРЕД {root['link']}  ({len(t)} msg)")
    print("--- было ---")
    print(_thread_text(t)[:500])
    for j in js:
        print(f"--- стало: [{j['type']}] city={j.get('city')} ---")
        print(f"  Q: {j['question']}")
        print(f"  A: {j['answer']}")
        print(f"--- судья: verdict={j['verdict']}  faithful={j['faithful']} atomic={j['atomic']} "
              f"useful={j['useful']} type_ok={j['type_ok']} (suggested: {j['type_suggested']}) "
              f"city_ok={j['city_ok']} (suggested: {j['city_suggested']})")
        if j["note"]:
            print(f"  note: {j['note']}")
    print()

# Сводка по этим 20 тредам:
from collections import Counter
print("verdict:", dict(Counter(j["verdict"] for j in judged)))


[eval:precision] 13/13
ТРЕД https://t.me/helpgeorgia/308018  (6 msg)
--- было ---
Alex: Ребят, а где в Тбилиси делается справка о несудимости? 🥸
Аноним: https://t.me/nlevshitstelegram/19384
Alex: А не подскажете, как записаться в сферу интересов РФ для оформления справки о несудимости именно в РФ? 🇷🇺
Аноним: Позвоните им) Правила могут меняться, так будет самое надёжное
Andrew: На госуслугах я заказывал эту справку. Никуда ехать не приходилось
Nadezhda P.: Апостиль был на справке ?
--- стало: [vote_based] ---
  Q: Где в Тбилиси делается справка о несудимости?
  A: Справка о несудимости в Тбилиси делается в соответствующих государственных учреждениях, информацию можно найти по ссылке.
--- судья: verdict=drop  faithful=False atomic=True useful=True type_ok=True (suggested: vote_based)
  note: Ответ не подтверждается тредом. В треде нет информации о том, где в Тбилиси делается справка о несудимости. Ссылка, предоставленная в ответе, не объяснена и не подтверждает информацию.
--- стало: [v

## 4e. Fix — apply the judge's verdicts

Turns eval_knowledge's verdicts into an actual fixed dataset: adopts the suggested type, re-splits non-atomic pairs, and for faithful=false drops gets a second opinion from `config.REVERIFY_MODEL` before discarding — only drops when both judges agree; a disagreement means the pair is kept (and counted separately as `rescued` so you can see which keeps came from a disagreement).

In [ ]:
# ⚠️ Spends OpenAI tokens (one call per atomize + one per faithful=false drop).
from src.fix_knowledge import fix_batch

result = fix_batch(knowledge, judged, threads_by_root)

print(f"kept: {len(result['kept'])}  (rescued by 2nd judge: {len(result['rescued'])})")
print(f"fixed: {len(result['fixed'])}")
print(f"dropped: {len(result['dropped'])}")

print("\n=== FIXED (type/city corrected / re-atomized) ===")
for u in result['fixed']:
    print(f"  [{u['type']}] city={u.get('city')} Q: {u['question']}")
    print(f"        A: {u['answer'][:150]}")

print("\n=== RESCUED (2nd judge disagreed → kept) ===")
for u in result['rescued']:
    print(f"  Q: {u['question']}")
    print(f"     1st judge (drop): {u['judge_note']}")
    print(f"     2nd judge (keep): {u['reverify_note']}")

print("\n=== DROPPED (both judges agreed unfaithful, or useful=false) ===")
for u in result['dropped']:
    print(f"  Q: {u['question']}")
    print(f"     reason: {u.get('drop_reason', '')}")
    if u.get('reverify_note'):
        print(f"     2nd judge confirmed: {u['reverify_note']}")


## 4f. Итоговые знания

`result['kept']` уже включает в себя `rescued` (спасённые вторым судьёй), поэтому финальный набор — это просто `kept + fixed`, без дублей.

In [ ]:
from src.fix_knowledge import save_fixed

final_knowledge = result['kept'] + result['fixed']
print(f"итог: {len(final_knowledge)} знаний из {len(knowledge)} исходных (distilled) / {len(preview_threads)} тредов\n")

for u in final_knowledge:
    print(f"[{u['type']}] city={u.get('city')} {u['question']}")
    print(f"  {u['answer']}")
    print(f"  ↳ {u['root_link']}")
    print()

save_fixed(chat['username'], result)


## 5. Indexing (embeddings → Chroma)

⚠️ This step spends OpenAI tokens (embeddings are cheap but not free). Run it once the chunks look good.

In [ ]:
# Indexing is usually more convenient to run as scripts (they read data/chunks/*.jsonl):
#   uv run python -m src.preprocess   # save chunks to disk
#   uv run python -m src.index        # build the index
#
# Or right here:
from src.index import main as build_index
from src.preprocess import main as build_chunks

build_chunks()   # data/chunks/*.jsonl
build_index()    # embeddings -> chroma_db/

In [ ]:
from src.store import get_collection
col = get_collection()
print("Records in collection:", col.count())

## 6. Retrieval

See which fragments are found for a question and how relevant they are (score closer to 1 = better).

In [ ]:
from src.retrieve import search

query = "как открыть ип в грузии"  # <- change the question (keep it in Russian)

for i, h in enumerate(search(query, k=5), 1):
    print(f"--- #{i}  score={h['score']:.3f}  {h['meta']['link']}")
    print(h["text"][:300])
    print()

## 7. RAG answer

The final answer from GPT based on the retrieved fragments + the list of sources.

In [ ]:
from src.rag import answer

res = answer("какие документы нужны для открытия ип")  # <- change the question

print(res["answer"])
print("\n——— Sources ———")
for s in res["sources"]:
    print(s["title"], "|", s["link"])

## 8. Debug: which prompt is actually sent to GPT

Useful to understand why the model answered the way it did, and to tweak the system prompt in `src/rag.py`.

In [ ]:
from src.rag import _build_context, SYSTEM_PROMPT
from src.retrieve import search

q = "как получить внж"
hits = search(q, k=4)

print("### SYSTEM PROMPT ###\n")
print(SYSTEM_PROMPT)
print("\n### CONTEXT (fragments) ###\n")
print(_build_context(hits))